# 🥑 Notebook 01: Exploratory Data Analysis & Data Cleaning
## Avocado Ripening Stage Classification — Greenpods Labs Assignment 2

**Objective:** Understand the dataset structure, identify class imbalance, detect outliers, and document biases before model training.

---

## 1. Environment Setup & Library Imports

In [ ]:
import os
import sys
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import load_image_paths_and_labels, CLASS_NAMES

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('Set2')

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
print(f'Project Root: {PROJECT_ROOT}')
print(f'Data Dir: {DATA_DIR}')


## 2. Dataset Download (Run only if data is not yet available)

In [ ]:

# from src.dataset import download_dataset
# download_dataset(

if DATA_DIR.exists():
    folders = [d.name for d in DATA_DIR.iterdir() if d.is_dir()]
    print(f'Available class folders: {folders}')
else:
    print(f'[WARNING] {DATA_DIR} does not exist. Please download the dataset.')


## 3. Dataset Structure & Class Distribution

In [ ]:
def analyze_avocado_dataset(data_dir):
    """Walk the dataset directory and collect class-level statistics."""
    data_dir = Path(data_dir)
    valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    
    class_stats = {}
    for cls_dir in sorted(data_dir.iterdir()):
        if cls_dir.is_dir():
            images = [f for f in cls_dir.iterdir() if f.suffix.lower() in valid_exts]
            sizes = []
            for img_path in images:
                try:
                    with Image.open(img_path) as img:
                        sizes.append(img.size)
                except Exception:
                    pass
            
            class_stats[cls_dir.name] = {
                'count': len(images),
                'paths': [str(p) for p in images],
                'sizes': sizes,
            }
    return class_stats

class_stats = analyze_avocado_dataset(DATA_DIR)

summary_data = []
for cls_name, stats in class_stats.items():
    if stats['sizes']:
        widths  = [s[0] for s in stats['sizes']]
        heights = [s[1] for s in stats['sizes']]
        summary_data.append({
            'Ripening Stage': cls_name.capitalize(),
            'Image Count': stats['count'],
            'Avg Width': int(np.mean(widths)),
            'Avg Height': int(np.mean(heights)),
            'Min W×H': f"{min(widths)}×{min(heights)}",
            'Max W×H': f"{max(widths)}×{max(heights)}",
        })

df_summary = pd.DataFrame(summary_data)
total = df_summary['Image Count'].sum()
df_summary['Proportion (%)'] = (df_summary['Image Count'] / total * 100).round(1)

print(f'\n{'='*65}')
print(f'  DATASET SUMMARY — Total Images: {total}')
print(f'{'='*65}')
print(df_summary.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Avocado Dataset — Class Distribution', fontsize=16, fontweight='bold', y=1.02)

stages = df_summary['Ripening Stage']
counts = df_summary['Image Count']
colors = ['#4CAF50', '#FFC107', '#FF7043', '#8D6E63']

bars = axes[0].bar(stages, counts, color=colors, edgecolor='white', linewidth=1.5, zorder=3)
axes[0].set_xlabel('Ripening Stage', fontsize=12)
axes[0].set_ylabel('Number of Images', fontsize=12)
axes[0].set_title('Image Count per Stage', fontsize=13)
axes[0].grid(axis='y', alpha=0.3, zorder=0)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(count), ha='center', va='bottom', fontweight='bold', fontsize=11)

wedges, texts, autotexts = axes[1].pie(
    counts, labels=stages, autopct='%1.1f%%',
    colors=colors, startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    pctdistance=0.75
)
for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight('bold')
axes[1].set_title('Class Proportion', fontsize=13)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data' / 'processed' / 'class_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Class distribution plot saved.')


## 4. Sample Image Visualisation per Class

In [ ]:
n_samples = 5
fig, axes = plt.subplots(len(class_stats), n_samples, figsize=(16, 3 * len(class_stats)))
fig.suptitle('Sample Images per Ripening Stage', fontsize=16, fontweight='bold')

if len(class_stats) == 1:
    axes = [axes]

for row_idx, (cls_name, stats) in enumerate(class_stats.items()):
    paths = stats['paths']
    samples = random.sample(paths, min(n_samples, len(paths)))
    
    for col_idx in range(n_samples):
        ax = axes[row_idx][col_idx]
        if col_idx < len(samples):
            img = Image.open(samples[col_idx]).resize((200, 200))
            ax.imshow(img)
            if col_idx == 0:
                ax.set_ylabel(cls_name.upper(), fontsize=11,
                              fontweight='bold', rotation=90, labelpad=10)
        else:
            ax.axis('off')
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data' / 'processed' / 'sample_images.png',
            dpi=120, bbox_inches='tight')
plt.show()


## 5. Color Space Analysis — CIELAB & HSV

In [ ]:
def plot_color_distributions(class_stats, n_per_class=30, save_path=None):
    """
    Plot L*, a*, b* channel histograms per class to visualise
    how skin color shifts across ripening stages.
    """
    fig, axes = plt.subplots(3, len(class_stats), figsize=(14, 9))
    fig.suptitle('CIELAB Channel Distributions per Ripening Stage',
                 fontsize=15, fontweight='bold')
    
    channel_labels = ['L* (Lightness)', 'a* (Green→Red)', 'b* (Blue→Yellow)']
    channel_colors = ['#607D8B', '#E57373', '#FFD54F']
    
    for col_idx, (cls_name, stats) in enumerate(class_stats.items()):
        paths = stats['paths'][:n_per_class]
        all_channels = [[], [], []]
        
        for p in paths:
            img_bgr = cv2.imread(p)
            if img_bgr is None:
                continue
            img_bgr = cv2.resize(img_bgr, (128, 128))
            lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
            for ch in range(3):
                all_channels[ch].extend(lab[:, :, ch].ravel().tolist())
        
        for row_idx in range(3):
            ax = axes[row_idx][col_idx]
            ax.hist(all_channels[row_idx], bins=50, color=channel_colors[row_idx],
                    alpha=0.75, edgecolor='none')
            if row_idx == 0:
                ax.set_title(cls_name.capitalize(), fontsize=12, fontweight='bold')
            if col_idx == 0:
                ax.set_ylabel(channel_labels[row_idx], fontsize=10)
            ax.grid(alpha=0.2)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

os.makedirs(PROJECT_ROOT / 'data' / 'processed', exist_ok=True)
plot_color_distributions(
    class_stats,
    save_path=PROJECT_ROOT / 'data' / 'processed' / 'lab_distributions.png'
)


## 6. Class Imbalance Analysis

In [ ]:
print('\n' + '='*55)
print('  CLASS IMBALANCE ANALYSIS')
print('='*55)

counts = {cls: stats['count'] for cls, stats in class_stats.items()}
max_count = max(counts.values())
min_count = min(counts.values())
imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')

print(f'  Max class count : {max_count}')
print(f'  Min class count : {min_count}')
print(f'  Imbalance ratio : {imbalance_ratio:.2f}x')

if imbalance_ratio > 3:
    print('\n  ⚠ SIGNIFICANT class imbalance detected!')
    print('  Mitigation strategies:')
    print('    • class_weight="balanced" in SVM / Random Forest')
    print('    • Weighted CrossEntropyLoss in DL training')
    print('    • Over-sampling minority classes with augmentation')
elif imbalance_ratio > 1.5:
    print('\n  ⚡ MILD imbalance detected. class_weight="balanced" recommended.')
else:
    print('\n  ✓ Dataset is reasonably balanced.')

fig, ax = plt.subplots(figsize=(9, 3.5))
cls_names = list(counts.keys())
cls_counts = list(counts.values())
bar_colors = ['#4CAF50', '#FFC107', '#FF7043', '#8D6E63'][:len(cls_names)]
bars = ax.barh(cls_names, cls_counts, color=bar_colors, edgecolor='white')
ax.set_xlabel('Image Count', fontsize=11)
ax.set_title('Class Balance Overview', fontsize=13, fontweight='bold')
for bar, cnt in zip(bars, cls_counts):
    ax.text(cnt + 2, bar.get_y() + bar.get_height()/2,
            str(cnt), va='center', fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Outlier & Quality Check

In [ ]:
def detect_outlier_images(class_stats, brightness_thresh_low=30, brightness_thresh_high=225):
    """
    Detect images that are too dark, too bright, or very small.
    These may indicate corrupted, mislabeled, or poor-quality samples.
    """
    outliers = []
    
    for cls_name, stats in class_stats.items():
        for img_path in stats['paths']:
            img = cv2.imread(img_path)
            if img is None:
                outliers.append({'path': img_path, 'class': cls_name, 'issue': 'CORRUPTED'})
                continue
            
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            mean_brightness = float(gray.mean())
            h, w = img.shape[:2]
            
            if mean_brightness < brightness_thresh_low:
                outliers.append({'path': img_path, 'class': cls_name,
                                 'issue': f'TOO DARK (brightness={mean_brightness:.1f})'})
            elif mean_brightness > brightness_thresh_high:
                outliers.append({'path': img_path, 'class': cls_name,
                                 'issue': f'TOO BRIGHT (brightness={mean_brightness:.1f})'})
            elif min(h, w) < 64:
                outliers.append({'path': img_path, 'class': cls_name,
                                 'issue': f'TOO SMALL ({w}×{h})'})
    
    return outliers

outliers = detect_outlier_images(class_stats)
print(f'Total outlier images detected: {len(outliers)}')
if outliers:
    df_outliers = pd.DataFrame(outliers)
    print(df_outliers.groupby(['class', 'issue']).size().reset_index(name='count'))
    print('\nFirst 5 outliers:')
    print(df_outliers.head().to_string(index=False))
else:
    print('✓ No significant quality outliers found.')


## 8. Dataset Biases & Observations

In [ ]:
bias_report = """
╔══════════════════════════════════════════════════════════════════╗
║           DATASET BIAS & LIMITATION REPORT                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. SKIN COLOR SHIFTS                                            ║
║     • Underripe: Bright green skin → high a* (green) in LAB     ║
║     • Breaking:  Green fading to olive/purple tones              ║
║     • Ripe:      Dark purple/black skin → low a* values          ║
║     • Overripe:  Black/brown wrinkled skin with decay spots      ║
║     → Ripe ↔ Overripe boundary is ambiguous (subjective labels)  ║
║                                                                  ║
║  2. LIGHTING ARTIFACTS                                           ║
║     • Specular highlights on shiny underripe skins               ║
║     • Shadow overlaps on dark ripe/overripe avocados             ║
║     → These create false brightness in L* channel                ║
║                                                                  ║
║  3. BACKGROUND VARIATION                                         ║
║     • Images shot on white, wooden, and dark surfaces            ║
║     → Background pixels pollute global color histograms          ║
║                                                                  ║
║  4. VIEWPOINT BIAS                                               ║
║     • Single-view images cannot capture hidden bruises           ║
║     → A single-view model may miss localised decay               ║
║                                                                  ║
║  5. VARIETY BIAS                                                 ║
║     • Some avocado varieties remain green even when ripe         ║
║     → Color-only models will misclassify these varieties         ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(bias_report)


## 9. Save Processed Metadata

In [ ]:
from src.dataset import load_image_paths_and_labels, stratified_split

paths, labels, found_classes = load_image_paths_and_labels(str(DATA_DIR))

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(paths, labels)

print('Dataset Split Summary:')
print(f'  Train: {len(X_train)} images')
print(f'  Val  : {len(X_val)} images')
print(f'  Test : {len(X_test)} images')
print(f'  Total: {len(paths)} images')

print(f'\nSplit Ratios:')
total = len(paths)
print(f'  Train: {len(X_train)/total*100:.1f}%')
print(f'  Val  : {len(X_val)/total*100:.1f}%')
print(f'  Test : {len(X_test)/total*100:.1f}%')

import json
split_info = {
    'class_names': found_classes,
    'total': len(paths),
    'train': len(X_train),
    'val': len(X_val),
    'test': len(X_test),
    'class_distribution': counts,
}
os.makedirs(PROJECT_ROOT / 'data' / 'processed', exist_ok=True)
with open(PROJECT_ROOT / 'data' / 'processed' / 'split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)
print('\nSplit info saved to data/processed/split_info.json')


## 10. EDA Summary

| Insight | Finding |
|---------|------|
| Classes | underripe, breaking, ripe, overripe |
| Class Imbalance | Check above — apply class_weight='balanced' if needed |
| Key Feature | a* channel in CIELAB (green→red transition) |
| Key Challenge | Ripe/Overripe boundary overlap |
| Outliers | Low-brightness, high-brightness, corrupted images |
| Augmentation | Rotation, flip, color jitter to address lighting variation |

---

**➡️ Next: `02_approach_1_classical_ml.ipynb`**